# Программирование для всех <br>(основы работы с Python)

*Алла Тамбовцева*

## Практикум 7*. Более продвинутый парсинг, обработка JSON и преобразование датафреймов

*Продолжение практикума 7, имеет смысл изучать после знакомства с `pandas` (создание столбцов и lambda-функции).*

Импортируем все необходимое:

In [1]:
import requests
import json
import time
import pandas as pd
from bs4 import BeautifulSoup

Сразу выключим предупреждения о проверки сертификатов (издержки `verify = False` в `requests`):

In [2]:
from urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(category=InsecureRequestWarning)

В практикуме 7 мы написали универсальную функцию для сайта nplus1.ru, которая принимает на вход ссылку на новость, а возвращает кортеж с текстом новости и другими ее атрибутами. Остается вопрос: а как быть, если мы хотим собрать новости за конкретный период? Не на всех сайтах это возможно сделать удобным образом, однако в данном случае на сайте есть удобные разделы по датам. Например, откроем новости за 27 октября 2026 года: https://nplus1.ru/news/2026/10/27.

Если мы посмотрим на страницу по ссылке выше, заметим, что на ней хранятся обычные ссылки на новости, как их собирать, мы уже знаем. Поэтому давайте поймем, как сформировать список ссылок с датами внутри.

### Сюжет 1. Форматирование строк и подстановка чисел в ссылки

На строках Python существует метод `.zfill()`, который добавляет недостающие нули в начало строки (столько, сколько нужно для определенного числа символов в строке):

In [3]:
# работа .zfill()
# в начало строки дописываются 0, чтобы получить нужное число символов

print("1".zfill(2))
print("1".zfill(3))
print("123".zfill(3))

01
001
123


Если нам нужно получить все ссылки за январь, мы можем запустить цикл (от 1 до 31), подставить число в f-строку со ссылкой и добавить 0 в начало, если число однозначное:

In [4]:
# не забываем превратить i в строку,
# .zfill() с числами не работает

january = []
for i in range(1, 32):
    day = str(i).zfill(2)
    url = f"https://nplus1.ru/news/2026/01/{day}"
    january.append(url)

print(january)

['https://nplus1.ru/news/2026/01/01', 'https://nplus1.ru/news/2026/01/02', 'https://nplus1.ru/news/2026/01/03', 'https://nplus1.ru/news/2026/01/04', 'https://nplus1.ru/news/2026/01/05', 'https://nplus1.ru/news/2026/01/06', 'https://nplus1.ru/news/2026/01/07', 'https://nplus1.ru/news/2026/01/08', 'https://nplus1.ru/news/2026/01/09', 'https://nplus1.ru/news/2026/01/10', 'https://nplus1.ru/news/2026/01/11', 'https://nplus1.ru/news/2026/01/12', 'https://nplus1.ru/news/2026/01/13', 'https://nplus1.ru/news/2026/01/14', 'https://nplus1.ru/news/2026/01/15', 'https://nplus1.ru/news/2026/01/16', 'https://nplus1.ru/news/2026/01/17', 'https://nplus1.ru/news/2026/01/18', 'https://nplus1.ru/news/2026/01/19', 'https://nplus1.ru/news/2026/01/20', 'https://nplus1.ru/news/2026/01/21', 'https://nplus1.ru/news/2026/01/22', 'https://nplus1.ru/news/2026/01/23', 'https://nplus1.ru/news/2026/01/24', 'https://nplus1.ru/news/2026/01/25', 'https://nplus1.ru/news/2026/01/26', 'https://nplus1.ru/news/2026/01/27', 

Чтобы написать код, который сформирует список ссылок на даты за весь 2026 год (31 ссылка на даты января, 28 ссылок на даты февраля и так далее), имеет смысл начать со списка с номерами месяцев и списка с числом дней в них.

In [5]:
# месяцы
months = range(1, 13)
print(*months)

# дни в каждом месяце, год не високосный
ndays = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
print(*ndays)

# пары
print(*zip(months, ndays))

1 2 3 4 5 6 7 8 9 10 11 12
31 28 31 30 31 30 31 31 30 31 30 31
(1, 31) (2, 28) (3, 31) (4, 30) (5, 31) (6, 30) (7, 31) (8, 31) (9, 30) (10, 31) (11, 30) (12, 31)


Теперь давайте пройдемся по парам в `zip()` и с помощью вложенного цикла будем подставлять в ссылку и месяц, и день сразу:

In [6]:
# сначала проходимся по месяцам, а внутри каждого
# месяца – по дням

year = []
for mon, n in zip(months, ndays):
    for date in range(1, n + 1):
        link_date = f"https://nplus1.ru/news/2026/{str(mon).zfill(2)}/{str(date).zfill(2)}"
        year.append(link_date)
        
print(len(year)) # 365 ссылок
print(year[-3:]) # последние 3 ссылки для проверки

365
['https://nplus1.ru/news/2026/12/29', 'https://nplus1.ru/news/2026/12/30', 'https://nplus1.ru/news/2026/12/31']


Список ссылок на даты готов!

### Сюжет 2. Формат JSON и его обработка в рамках блока кода JavaScript

Попробуем отправить запрос к странице, на которой хранятся новости за определенную дату и найти все ссылки на странице:

In [7]:
url = "https://nplus1.ru/news/2026/11/03"
soup = BeautifulSoup(requests.get(url, verify = False).text)

# раскомментируйте строку ниже
# soup.find_all("a")

Проблема! Ссылки на странице есть (это не «пустая» страница, возникшая из-за капчи или отказа в доступе), однако в исходном коде, который забирается как текстовый слой, их нет. Вместо этого в исходном коде страницы есть блок кода на JavaScript, который «подтягивает» на эту страницу не только ссылки на новости, но и их характеристики: заголовок, сложность, дату и рубрики.

Код JavaScript обычно заключается в тэг `<script></script>`. Найдем все и выберем (просто визуально, методом проб и ошибок) нужный:

In [8]:
js = soup.find_all("script")[3].text
# print(js)

**NB.** Если после исполнения кода выше в `js` пустая строка с текстом – замените `.text` на `.next`, это особенность другой версии `bs4`:

    soup.find_all("script")[3].next
    
Так как формально внутри `<script>` не текст, а код JavaScript, текст через атрибут `.text` в более новой версии `bs4` не находится (более строгое отношение к типам). А атрибут `.next` возвращает содержимое, следующее за тэгом, что в нашем случае – ровно то, что нужно.

Объект `js` сейчас – это обычная строка (текст) с кодом на языке JavaScript. Знать этот язык нам совсем не обязательно, достаточно понять, какой фрагмент текста содержит ссылки. В данном случае текст в `js` выглядит жутко, это из-за того, что двойные кавычки (`"`) местами представлены в виде символов Unicode. Выполним замену, чтобы меньше пугаться:

In [9]:
js = js.replace("\\u0022", '"')
# js

Чтобы еще меньше пугаться, давайте заменим блоки из `\` на пустой символ:

In [10]:
js = js.replace("\\\\\\", "")
# js

В целом, уже сейчас можно попытаться найти в `js` фрагменты со ссылками на новости, но это неудобно. Поэтому давайте выберем текст после `JSON.parse(` и попробуем привести его к «чистой» JSON-строке!

In [11]:
# разбиваем по JSON.parse(\' и забираем все, что после (индекс 1)
# метод .strip() без текста внутри убирает отступы в начале/конце строки,
# а с текстом внутри – убирает этот текст в начале/конце строки

json_raw = js.split("JSON.parse(\'")[1].strip().strip("\');")

Если раскомментируете строку ниже, увидите валидную строку JSON с началом и концом как у словаря (строго фигурные скобки без лишних символов вокруг).

In [ ]:
# json_raw

Теперь эту строку можно десериализовать – превратить в словарь!

In [12]:
D = json.loads(json_raw)
# print(D)

Давайте выберем из словаря запись с нужными данными и превратим их в таблицу!

In [13]:
df = pd.DataFrame(D["data"])
df

,type,title,typeKey,isImage,difficulty,themes,url,imageBg,rubrics,isVisibleTime,author,date,image,imageBox,time,titleClear,subtitle
0,default,\u0410\u0444\u0430\u043b\u0438\u043d\u0430 \u0...,news,True,"{'link': '/material/difficulty/1', 'value': 1....",[],https://nplus1.ru/news/2026/11/03/dolphin-sets...,#ffffff,"[{'id': 842, 'name': '\u0417\u043e\u043e\u043b...",False,"{'id': 222, 'name': '\u0421\u0435\u0440\u0433\...","{'link': '/news/2026/11/03', 'value': '03.11.2...",None,"{'original': {'hex': None, 'src': 'https://min...","{'link': '/search/empty/', 'value': '18:59', '...",None,\u042d\u0442\u043e \u043e\u0431\u0435\u0441\u0...
1,default,\u0412\u0430\u043a\u0446\u0438\u043d\u0430\u04...,news,True,"{'link': '/material/difficulty/1', 'value': 1....",[],https://nplus1.ru/news/2026/11/03/covid-vaccin...,#ffffff,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...",False,"{'id': 225, 'name': '\u0421\u043b\u0430\u0432\...","{'link': '/news/2026/11/03', 'value': '03.11.2...",None,"{'original': {'hex': None, 'src': 'https://min...","{'link': '/search/empty/', 'value': '15:54', '...",None,\u0412\u00a0\u0432\u043e\u0437\u0440\u0430\u04...
2,default,\u0412\u00a02023 \u0433\u043e\u0434\u0443 \u04...,news,True,"{'link': '/material/difficulty/2', 'value': 2....",[],https://nplus1.ru/news/2026/11/03/cervical-can...,#ffffff,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...",False,"{'id': 225, 'name': '\u0421\u043b\u0430\u0432\...","{'link': '/news/2026/11/03', 'value': '03.11.2...",None,"{'original': {'hex': None, 'src': 'https://min...","{'link': '/search/empty/', 'value': '14:22', '...",None,\u0411\u043e\u043b\u044c\u0448\u0435 \u0432\u0...
3,default,\u0411\u044b\u0432\u0448\u0438\u0435 \u043b\u0...,news,True,"{'link': '/material/difficulty/2', 'value': 2....",[],https://nplus1.ru/news/2026/11/03/nfl-players-...,#ffffff,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...",False,"{'id': 225, 'name': '\u0421\u043b\u0430\u0432\...","{'link': '/news/2026/11/03', 'value': '03.11.2...",None,"{'original': {'hex': None, 'src': 'https://min...","{'link': '/search/empty/', 'value': '12:42', '...",None,\u041e\u043d\u0438 \u043d\u0430\u0431\u043b\u0...
4,default,\u041c\u0435\u0442\u043a\u0438 \u043d\u0430\u0...,news,True,"{'link': '/material/difficulty/3', 'value': 3....",[],https://nplus1.ru/news/2026/11/03/luminescent-...,#ffffff,"[{'id': 838, 'name': '\u0425\u0438\u043c\u0438...",False,"{'id': 529, 'name': '\u0412\u043b\u0430\u0434\...","{'link': '/news/2026/11/03', 'value': '03.11.2...",None,"{'original': {'hex': None, 'src': 'https://min...","{'link': '/search/empty/', 'value': '12:16', '...",None,\u0418\u0445\u00a0\u043f\u0440\u043e\u0432\u04...
5,default,\u0421\u0442\u0430\u0440\u0438\u043d\u043d\u04...,news,True,"{'link': '/material/difficulty/2', 'value': 2....",[],https://nplus1.ru/news/2026/11/03/gizhiginsky-...,#ffffff,"[{'id': 844, 'name': '\u0410\u0440\u0445\u0435...",False,"{'id': 275, 'name': '\u041c\u0438\u0445\u0430\...","{'link': '/news/2026/11/03', 'value': '03.11.2...",None,"{'original': {'hex': None, 'src': 'https://min...","{'link': '/search/empty/', 'value': '10:47', '...",None,\u041f\u0440\u0438 \u044d\u0442\u043e\u043c \u...
6,default,\u0417\u043e\u043e\u043b\u043e\u0433\u0438 \u0...,news,True,"{'link': '/material/difficulty/3', 'value': 3....",[],https://nplus1.ru/news/2026/11/03/achille-and-...,#ffffff,"[{'id': 842, 'name': '\u0417\u043e\u043e\u043b...",False,"{'id': 222, 'name': '\u0421\u0435\u0440\u0433\...","{'link': '/news/2026/11/03', 'value': '03.11.2...",None,"{'original': {'hex': None, 'src': 'https://min...","{'link': '/search/empty/', 'value': '10:34', '...",None,\u0410\u0445\u0438\u043b\u043b \u0438\u00a0\u0...


### Сюжет 3. Обработка датафрейма со словарями внутри и перекодирование Unicode

Выберем из таблицы нужные столбцы:

In [14]:
data = df[["title", "difficulty", "url", 
           "rubrics", "author", "date"]]
data

,title,difficulty,url,rubrics,author,date
0,\u0410\u0444\u0430\u043b\u0438\u043d\u0430 \u0...,"{'link': '/material/difficulty/1', 'value': 1....",https://nplus1.ru/news/2026/11/03/dolphin-sets...,"[{'id': 842, 'name': '\u0417\u043e\u043e\u043b...","{'id': 222, 'name': '\u0421\u0435\u0440\u0433\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
1,\u0412\u0430\u043a\u0446\u0438\u043d\u0430\u04...,"{'link': '/material/difficulty/1', 'value': 1....",https://nplus1.ru/news/2026/11/03/covid-vaccin...,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...","{'id': 225, 'name': '\u0421\u043b\u0430\u0432\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
2,\u0412\u00a02023 \u0433\u043e\u0434\u0443 \u04...,"{'link': '/material/difficulty/2', 'value': 2....",https://nplus1.ru/news/2026/11/03/cervical-can...,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...","{'id': 225, 'name': '\u0421\u043b\u0430\u0432\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
3,\u0411\u044b\u0432\u0448\u0438\u0435 \u043b\u0...,"{'link': '/material/difficulty/2', 'value': 2....",https://nplus1.ru/news/2026/11/03/nfl-players-...,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...","{'id': 225, 'name': '\u0421\u043b\u0430\u0432\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
4,\u041c\u0435\u0442\u043a\u0438 \u043d\u0430\u0...,"{'link': '/material/difficulty/3', 'value': 3....",https://nplus1.ru/news/2026/11/03/luminescent-...,"[{'id': 838, 'name': '\u0425\u0438\u043c\u0438...","{'id': 529, 'name': '\u0412\u043b\u0430\u0434\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
5,\u0421\u0442\u0430\u0440\u0438\u043d\u043d\u04...,"{'link': '/material/difficulty/2', 'value': 2....",https://nplus1.ru/news/2026/11/03/gizhiginsky-...,"[{'id': 844, 'name': '\u0410\u0440\u0445\u0435...","{'id': 275, 'name': '\u041c\u0438\u0445\u0430\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
6,\u0417\u043e\u043e\u043b\u043e\u0433\u0438 \u0...,"{'link': '/material/difficulty/3', 'value': 3....",https://nplus1.ru/news/2026/11/03/achille-and-...,"[{'id': 842, 'name': '\u0417\u043e\u043e\u043b...","{'id': 222, 'name': '\u0421\u0435\u0440\u0433\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."


Пора избавиться от Unicode и увидеть текст на кириллице! Рассмотрим пример перекодирования на маленькой строке с текстом:

In [15]:
# метод .encode() переводит строку (тип str) в байты (тип bytes)
# метод .decode() байты в строку с заданным форматом

s = "\\u0417\\u043e\\u043e".encode().decode('unicode_escape')
print(s)

Зоо


Теперь выберем столбец с заголовком и применим ко всем ячейкам lambda-функцию, которая выполнит перекодирование:

In [16]:
data["title"] = data["title"].apply(lambda x: x.encode().decode('unicode_escape'))
data

,title,difficulty,url,rubrics,author,date
0,Афалина губкой перекрыла рыбам вход в убежище,"{'link': '/material/difficulty/1', 'value': 1....",https://nplus1.ru/news/2026/11/03/dolphin-sets...,"[{'id': 842, 'name': '\u0417\u043e\u043e\u043b...","{'id': 222, 'name': '\u0421\u0435\u0440\u0433\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
1,Вакцинация от ковида во время беременности не ...,"{'link': '/material/difficulty/1', 'value': 1....",https://nplus1.ru/news/2026/11/03/covid-vaccin...,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...","{'id': 225, 'name': '\u0421\u043b\u0430\u0432\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
2,В 2023 году зафиксировали 887 тысяч новых случ...,"{'link': '/material/difficulty/2', 'value': 2....",https://nplus1.ru/news/2026/11/03/cervical-can...,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...","{'id': 225, 'name': '\u0421\u043b\u0430\u0432\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
3,Бывшие линейные игроки в американский футбол с...,"{'link': '/material/difficulty/2', 'value': 2....",https://nplus1.ru/news/2026/11/03/nfl-players-...,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...","{'id': 225, 'name': '\u0421\u043b\u0430\u0432\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
4,Метки на основе борной кислоты поменяли яркост...,"{'link': '/material/difficulty/3', 'value': 3....",https://nplus1.ru/news/2026/11/03/luminescent-...,"[{'id': 838, 'name': '\u0425\u0438\u043c\u0438...","{'id': 529, 'name': '\u0412\u043b\u0430\u0434\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
5,Старинного шамана из Магаданской области назва...,"{'link': '/material/difficulty/2', 'value': 2....",https://nplus1.ru/news/2026/11/03/gizhiginsky-...,"[{'id': 844, 'name': '\u0410\u0440\u0445\u0435...","{'id': 275, 'name': '\u041c\u0438\u0445\u0430\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."
6,Зоологи описали однополую сексуальную связь бы...,"{'link': '/material/difficulty/3', 'value': 3....",https://nplus1.ru/news/2026/11/03/achille-and-...,"[{'id': 842, 'name': '\u0417\u043e\u043e\u043b...","{'id': 222, 'name': '\u0421\u0435\u0440\u0433\...","{'link': '/news/2026/11/03', 'value': '03.11.2..."


В остальных столбцах ситуация более сложная, надо разбирать содержимое одной ячейки на части. Посмотрим на содержимое первой ячейки:

* в столбце `difficulty`
* в столбце `date`
* в столбце `author`

In [17]:
print(data["difficulty"][0])
print(data["date"][0])
print(data["author"][0])

{'link': '/material/difficulty/1', 'value': 1.7, 'type': 'difficulty'}
{'link': '/news/2026/11/03', 'value': '03.11.26', 'value_itemprop': '2026-11-03', 'type': 'date'}
{'id': 222, 'name': '\\u0421\\u0435\\u0440\\u0433\\u0435\\u0439 \\u041a\\u043e\\u043b\\u0435\\u043d\\u043e\\u0432', 'position': '\\u0420\\u0435\\u0434\\u0430\\u043a\\u0442\\u043e\\u0440', 'avatar': 'https://minio.nplus1.ru/app-images/731349/524037edad17ee2e11d5292cbeb0b833.jpg?v=edcbabcd9c560cc18b90e14c0166b0544c5c4ba5ac1ae240a917578f2823e61d', 'link': '/author/sergei-kolenov', 'suffix': None, 'public_email': None, 'email': 'sergey-k-0@yandex.ru'}


Внутри ячейки – словари, можем забрать из каждого запись по соответствующему ключу:

In [18]:
print(data["difficulty"][0].get("value"))
print(data["date"][0].get("value"))

# автора еще перекодируем, он на кириллице

print(data["author"][0].get("name").encode().decode('unicode_escape'))

1.7
03.11.26
Сергей Коленов


Теперь применим lambda-функцию – «растянем» операции выше на все ячейки в столбцах:

In [19]:
data["difficulty"] = data["difficulty"].apply(lambda x: x.get("value"))
data["date"] = data["date"].apply(lambda x: x.get("value"))
data["author"] = data["author"].apply(lambda x: x.get("name").encode().decode('unicode_escape'))
data

,title,difficulty,url,rubrics,author,date
0,Афалина губкой перекрыла рыбам вход в убежище,1.7,https://nplus1.ru/news/2026/11/03/dolphin-sets...,"[{'id': 842, 'name': '\u0417\u043e\u043e\u043b...",Сергей Коленов,03.11.26
1,Вакцинация от ковида во время беременности не ...,1.9,https://nplus1.ru/news/2026/11/03/covid-vaccin...,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...",Слава Гоменюк,03.11.26
2,В 2023 году зафиксировали 887 тысяч новых случ...,2.1,https://nplus1.ru/news/2026/11/03/cervical-can...,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...",Слава Гоменюк,03.11.26
3,Бывшие линейные игроки в американский футбол с...,2.1,https://nplus1.ru/news/2026/11/03/nfl-players-...,"[{'id': 841, 'name': '\u041c\u0435\u0434\u0438...",Слава Гоменюк,03.11.26
4,Метки на основе борной кислоты поменяли яркост...,3.8,https://nplus1.ru/news/2026/11/03/luminescent-...,"[{'id': 838, 'name': '\u0425\u0438\u043c\u0438...",Владимир Кабанов,03.11.26
5,Старинного шамана из Магаданской области назва...,2.5,https://nplus1.ru/news/2026/11/03/gizhiginsky-...,"[{'id': 844, 'name': '\u0410\u0440\u0445\u0435...",Михаил Подрезов,03.11.26
6,Зоологи описали однополую сексуальную связь бы...,3.1,https://nplus1.ru/news/2026/11/03/achille-and-...,"[{'id': 842, 'name': '\u0417\u043e\u043e\u043b...",Сергей Коленов,03.11.26


Остались рубрики. Изучаем одну ячейку:

In [20]:
print(data["rubrics"][0])

[{'id': 842, 'name': '\\u0417\\u043e\\u043e\\u043b\\u043e\\u0433\\u0438\\u044f', 'link': 'https://nplus1.ru/search?tags=874', 'type': 'rubric'}]


Внутри – список со словарями, один словарь описывает одну рубрику. В данном случае рубрика одна, поэтому и словарь в списке один. Однако мы знаем, что рубрик может быть несколько. Чтобы их не потерять, нужен цикл или списковое включение:

In [21]:
# для наглядности сохраним ячейку в x
# проходим по всем словарям внутри через for и извлекаем
# название рубрики по ключу name, плюс, перекодируем

x = data["rubrics"][0]
print([i.get("name").encode().decode('unicode_escape') for i in x])

['Зоология']


Чтобы не писать длинную строку с функцией, напишем ее отдельно (и добавим склеивание рубрик из полученного списка в одну строку с перечнем рубрик):

In [22]:
def get_rubrics(x):
    rubs_list = [i.get("name").encode().decode('unicode_escape') for i in x]
    rubs_str = ", ".join(rubs_list)
    return rubs_str

data["rubrics"] = data["rubrics"].apply(get_rubrics)
data

,title,difficulty,url,rubrics,author,date
0,Афалина губкой перекрыла рыбам вход в убежище,1.7,https://nplus1.ru/news/2026/11/03/dolphin-sets...,Зоология,Сергей Коленов,03.11.26
1,Вакцинация от ковида во время беременности не ...,1.9,https://nplus1.ru/news/2026/11/03/covid-vaccin...,Медицина,Слава Гоменюк,03.11.26
2,В 2023 году зафиксировали 887 тысяч новых случ...,2.1,https://nplus1.ru/news/2026/11/03/cervical-can...,Медицина,Слава Гоменюк,03.11.26
3,Бывшие линейные игроки в американский футбол с...,2.1,https://nplus1.ru/news/2026/11/03/nfl-players-...,Медицина,Слава Гоменюк,03.11.26
4,Метки на основе борной кислоты поменяли яркост...,3.8,https://nplus1.ru/news/2026/11/03/luminescent-...,Химия,Владимир Кабанов,03.11.26
5,Старинного шамана из Магаданской области назва...,2.5,https://nplus1.ru/news/2026/11/03/gizhiginsky-...,Археология,Михаил Подрезов,03.11.26
6,Зоологи описали однополую сексуальную связь бы...,3.1,https://nplus1.ru/news/2026/11/03/achille-and-...,Зоология,Сергей Коленов,03.11.26


Все супер! Теперь надо написать универсальную функцию, которая будет принимать на вход ссылку с датой, а возвращать – датафрейм с очищенной информацией. Соберем весь код ранее:

In [23]:
# для удобства дублируем функцию, чтобы не потерять

def get_rubrics(x):
    rubs_list = [i.get("name").encode().decode('unicode_escape') for i in x]
    rubs_str = ", ".join(rubs_list)
    return rubs_str

# а вот новая

def get_meta_info(url):
    soup = BeautifulSoup(requests.get(url, verify =  False).text)
    js = soup.find_all("script")[3].text
    js = js.replace("\\u0022", '"').replace("\\\\\\", "")
    json_raw = js.split("JSON.parse(\'")[1].strip().strip("\');")
    D = json.loads(json_raw)
    
    df = pd.DataFrame(D["data"])
    data = df[["title", "difficulty", "url", 
               "rubrics", "author", "date"]]
    data["title"] = data["title"].apply(lambda x: 
                                        x.encode().decode('unicode_escape'))
    data["difficulty"] = data["difficulty"].apply(lambda x: x.get("value"))
    data["date"] = data["date"].apply(lambda x: x.get("value"))
    data["author"] = data["author"].apply(lambda x: 
                                          x.get("name").encode().decode('unicode_escape'))
    data["rubrics"] = data["rubrics"].apply(get_rubrics)
    return data

Проверяем работу функции на другой ссылке:

In [24]:
get_meta_info("https://nplus1.ru/news/2026/11/12")

,title,difficulty,url,rubrics,author,date
0,Google DeepMind представила модель для предска...,5.3,https://nplus1.ru/news/2026/11/12/alphagenome2,Медицина,Олег Лищук,12.11.26
1,В сточных водах университета во время экзамено...,2.4,https://nplus1.ru/news/2026/11/12/stress-in-wa...,Медицина,Олег Лищук,12.11.26
2,Кошки замяукали в присутствии хозяина и при ви...,1.5,https://nplus1.ru/news/2026/11/12/social-use-o...,Зоология,Катерина Петрова,12.11.26
3,Отмена гипотензивных препаратов утром перед оп...,2.2,https://nplus1.ru/news/2026/11/12/antihyperten...,Медицина,Слава Гоменюк,12.11.26
4,Положение при мочеиспускании у мужчин существе...,2.1,https://nplus1.ru/news/2026/11/12/standing-or-...,Медицина,Слава Гоменюк,12.11.26
5,При активных кластерных головных болях участил...,1.3,https://nplus1.ru/news/2026/11/12/cluster-burnout,Медицина,Олег Лищук,12.11.26
6,Фторирование воды не повлияло на когнитивные ф...,1.4,https://nplus1.ru/news/2026/11/12/fluoridated-...,Экология и климат,Олег Лищук,12.11.26
7,Алгоритм справился с поиском планетарных туман...,3.8,https://nplus1.ru/news/2026/11/12/pn-new-candi...,Астрономия,Александр Войтюк,12.11.26
8,Миниатюрные клопы нового вида провели всю жизн...,3.7,https://nplus1.ru/news/2026/11/12/heissophila-...,Зоология,Сергей Коленов,12.11.26
9,Деформация грудной стенки оказалась проявление...,2.1,https://nplus1.ru/news/2026/11/12/lung-hernia,Медицина,Слава Гоменюк,12.11.26


Все супер! Единственное, что огорчает – когда новости выгружаются на страницу с определенной датой, у них остается только одна рубрика, которая указана первой. Это не проблема нашего кода, это особенность сайта (сравните [карточку](https://nplus1.ru/news/2026/11/12) с новостью про Google DeepMind в «архиве» и на отдельной [странице](https://nplus1.ru/news/2026/11/12/alphagenome2) с текстом). Но мы все равно подстраховались :)

Оставим пока эту проблему и доработаем функцию так, чтобы она не ломалась в случае, если новостей нет (такое бывает, откройте страницу для 2026-01-01, например). Для единообразия поступим так: 

* если в `D["data"]` ничего нет (пустой список), создаем пустой датафрейм со столбцами `title`, `difficulty`, `url`,  `rubrics`, `author`, `date`;
* иначе создаем датафрейм со столбцами `title`, `difficulty`, `url`,  `rubrics`, `author`, `date` по старой схеме.

Чем такой подход хорош? Мы в любом случае получаем таблицу с фиксированным числом столбцов, а значит, при запуске функции в цикле мы сможем собрать все и склеить в единый датафрейм с данными за весь год!

In [25]:
def get_meta_info(url):
    soup = BeautifulSoup(requests.get(url, verify =  False).text)
    js = soup.find_all("script")[3].text
    js = js.replace("\\u0022", '"').replace("\\\\\\", "")
    json_raw = js.split("JSON.parse(\'")[1].strip().strip("\');")
    D = json.loads(json_raw)

    # для удобства сохраняем названия столбцов, чтобы не дублировать

    names = ["title", "difficulty", "url", 
             "rubrics", "author", "date"]

    if len(D["data"]) == 0:
        data = pd.DataFrame(columns = names)
    else:
        df = pd.DataFrame(D["data"])
        data = df[names]
        data["title"] = data["title"].apply(lambda x: 
                                            x.encode().decode('unicode_escape'))
        data["difficulty"] = data["difficulty"].apply(lambda x: x.get("value"))
        data["date"] = data["date"].apply(lambda x: x.get("value"))
        data["author"] = data["author"].apply(lambda x: 
                                              x.get("name").encode().decode('unicode_escape'))
        data["rubrics"] = data["rubrics"].apply(get_rubrics)
    return data

Проверяем на новостях от 1 января:

In [26]:
get_meta_info("https://nplus1.ru/news/2026/01/01") # ок

,title,difficulty,url,rubrics,author,date


Теперь можем написать цикл с задержкой по времени:

In [27]:
# для иллюстрации берем не целый год, а несколько дней ноября
# из списка year забираем несколько элементов

tables = []

for li in year[-60:-56]:
    df_res = get_meta_info(li)
    tables.append(df_res)
    time.sleep(1.5)

# склеиваем список таблиц в одну таблицу

final = pd.concat(tables)
final

,title,difficulty,url,rubrics,author,date
0,Глубоководную рыбу с прозрачной головой сняли ...,1.7,https://nplus1.ru/news/2026/11/02/macropinna-m...,Зоология,Сергей Коленов,02.11.26
1,Потерю родителя до 12 лет связали с повышенным...,2.3,https://nplus1.ru/news/2026/11/02/parental-dea...,Медицина,Слава Гоменюк,02.11.26
2,В нейроинтерфейс Neuralink встроили чат-бот Grok,1.4,https://nplus1.ru/news/2026/11/02/neuralink-grok,Медицина,Слава Гоменюк,02.11.26
3,Четверть американцев симулировала симптомы бол...,1.8,https://nplus1.ru/news/2026/11/02/symptom-misr...,Медицина,Слава Гоменюк,02.11.26
0,Афалина губкой перекрыла рыбам вход в убежище,1.7,https://nplus1.ru/news/2026/11/03/dolphin-sets...,Зоология,Сергей Коленов,03.11.26
1,Вакцинация от ковида во время беременности не ...,1.9,https://nplus1.ru/news/2026/11/03/covid-vaccin...,Медицина,Слава Гоменюк,03.11.26
2,В 2023 году зафиксировали 887 тысяч новых случ...,2.1,https://nplus1.ru/news/2026/11/03/cervical-can...,Медицина,Слава Гоменюк,03.11.26
3,Бывшие линейные игроки в американский футбол с...,2.1,https://nplus1.ru/news/2026/11/03/nfl-players-...,Медицина,Слава Гоменюк,03.11.26
4,Метки на основе борной кислоты поменяли яркост...,3.8,https://nplus1.ru/news/2026/11/03/luminescent-...,Химия,Владимир Кабанов,03.11.26
5,Старинного шамана из Магаданской области назва...,2.5,https://nplus1.ru/news/2026/11/03/gizhiginsky-...,Археология,Михаил Подрезов,03.11.26


Отлично, у нас есть способ получить мета-информацию (все, кроме текста) по каждой новости, и этот код можно адаптировать под любой год! Чтобы забрать текст новости и более подробный перечень рубрик, можно воспользоваться функцией `get_news()`из практикума 7. Для этого ее достаточно применить ко всем ссылкам на новости, а ссылки на новости можно получить из столбца `url` датафрейма выше.

P.S. При склеивании таблиц индексы строк автоматически не меняются, поэтому в данном случае номера строк перестали быть уникальными. Если дальше данные просто выгружать в файл, это не вызовет проблем, можно просто «выключить» сохранение индексов:

    final.to_csv("nplus1.csv", index = False)

При чтении этого CSV-файла в Python строки пронумеруются с 0, как обычно, без повторов. Если перед сохранением в файл потребуются манипуляции со строками, можно в атрибут `.index` записать новые значения:

    final.index = range(0, len(final))